# Speech Disorder Detection Using Machine Learning
**B.Tech Major Project (IV B.Tech I Sem)**  
*Department of Information Technology*  

### Objective
Analyze speech recordings to extract acoustic features (MFCCs, Pitch F0, RMS Energy, Zero-Crossing Rate, Spectral Centroid) and classify them into speech disorder categories (Dysarthria, Dysphonia, Stuttering vs. Normal) using Scikit-Learn classifiers (SVM, Random Forest, Logistic Regression).

---

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

# Add project root to sys.path
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import preprocess_audio
from src.feature_extraction import extract_feature_vector

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded successfully.')

## 1. Acoustic Waveform & Spectrogram Analysis across Disorder Categories

In [ ]:
classes = ['normal', 'dysarthria', 'dysphonia', 'stuttering']
fig, axes = plt.subplots(4, 2, figsize=(16, 12))

for idx, cls_name in enumerate(classes):
    sample_path = f'../data/{cls_name}/sample_{cls_name}_01.wav'
    if os.path.exists(sample_path):
        y, sr = preprocess_audio(sample_path)
        
        # Plot Waveform
        librosa.display.waveshow(y, sr=sr, ax=axes[idx, 0], color='#38bdf8')
        axes[idx, 0].set_title(cls_name.upper() + ' - Waveform Amplitude')
        axes[idx, 0].set_xlabel('Time (s)')
        
        # Plot Spectrogram (STFT dB)
        D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
        img = librosa.display.specshow(D, y_axis='log', x_axis='time', sr=sr, ax=axes[idx, 1], cmap='magma')
        axes[idx, 1].set_title(cls_name.upper() + ' - Log Spectrogram')

plt.tight_layout()
plt.show()

## 2. Feature Matrix Loading and Distribution Analysis

In [ ]:
features_path = '../data/features.csv'
df = pd.read_csv(features_path)
print(f'Loaded feature dataset: {df.shape[0]} samples, {df.shape[1]} columns.')
df.head()

In [ ]:
# Acoustic comparison across classes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.boxplot(data=df, x='label', y='pitch_mean', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Mean Pitch (F0 in Hz) by Speech Category')

sns.boxplot(data=df, x='label', y='rms_mean', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('RMS Energy by Speech Category')

sns.boxplot(data=df, x='label', y='zcr_mean', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Zero-Crossing Rate (ZCR) by Speech Category')

sns.boxplot(data=df, x='label', y='spectral_centroid_mean', ax=axes[1, 1], palette='Set2')
axes[1, 1].set_title('Spectral Centroid (Hz) by Speech Category')

plt.tight_layout()
plt.show()

## 3. Model Training & Comparative Performance

In [ ]:
from src.train import train_models
metadata = train_models(data_dir='../data', features_csv='../data/features.csv', models_dir='../models')

# Summarize benchmark results
benchmark_df = pd.DataFrame(metadata['models_benchmark']).T[['accuracy', 'precision', 'recall', 'f1_score']]
benchmark_df

In [ ]:
# Confusion Matrix Visualization
best_m = metadata['best_model']
cm = np.array(metadata['models_benchmark'][best_m]['confusion_matrix'])
classes = metadata['classes']

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title(f'Confusion Matrix - {best_m}')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.show()